# Стратифицированная рандомизация

Учебный эксперимент: сравниваем обычную и стратифицированную рандомизацию.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_users = 10_000

df = pd.DataFrame({
    "user_id": np.arange(1, n_users + 1),
    "os": np.random.choice(
        ["Android", "iOS"],
        size=n_users,
        p=[0.7, 0.3]
    )
})

df.head()

In [ ]:
df["os"].value_counts(normalize=True)

In [ ]:
df["rpu"] = np.where(
    df["os"] == "Android",
    np.random.normal(20, 5, n_users),
    np.random.normal(100, 10, n_users)
)

In [ ]:
df.groupby("os")["rpu"].agg(["count", "mean", "var", "std"])

## Рандомизация по OS

In [ ]:
df['group'] = 'Control' # Заполняем все строки Control

test_idx = (
    df.groupby('os')
      .sample(frac=0.5, random_state=42)
      .index
)

''' 
  1. Разделяем по os
  2. Случайная рандомизация внутри каждой группы frac=0.5 50%
  3. Получаемя index-сы пользоваателей
'''

df.loc[test_idx, 'group'] = 'Test'

In [ ]:
pd.crosstab(
    df["os"],
    df["group"]
)

In [ ]:
pd.crosstab(
    df["os"],
    df["group"],
    normalize="index"
)

In [ ]:
pd.crosstab(
    df["group"],
    df["os"],
    normalize="index"
)

In [ ]:
df["rpu_test"] = df["rpu"]

df.loc[df["group"] == "Test", "rpu_test"] *= 1.05

## Анализ стратифицированной рандомизации

In [ ]:
def analyze_ab(data, metric_col="rpu_test"):
    stats = data.groupby("group")[metric_col].agg(
        ["count", "mean", "var", "std"]
    )

    stats["var_mean"] = stats["var"] / stats["count"]

    se = (
        stats.loc["Test", "var_mean"]
        + stats.loc["Control", "var_mean"]
    ) ** 0.5

    delta = stats.loc["Test", "mean"] - stats.loc["Control", "mean"]
    t_stat = delta / se
    ci_left = delta - 1.96 * se
    ci_right = delta + 1.96 * se

    return stats, delta, se, t_stat, (ci_left, ci_right)

In [ ]:
strat, delta_strat, se_strat, t_stat_strat, ci_strat = analyze_ab(df)

print(strat)
print("SE", se_strat)
print("delta", delta_strat)
print("t-stat", t_stat_strat)
print("95% CI", ci_strat)

## Без стратиикации

In [ ]:
df_unstrat = df[["user_id", "os", "rpu"]].copy()

df_unstrat["group"] = "Control"

test_idx = (
    df_unstrat
    .sample(frac=0.5, random_state=42)
    .index
)

df_unstrat.loc[test_idx, "group"] = "Test"

df_unstrat["rpu_test"] = df_unstrat["rpu"]
df_unstrat.loc[df_unstrat["group"] == "Test", "rpu_test"] *= 1.05

In [ ]:
pd.crosstab(
    df_unstrat["group"],
    df_unstrat["os"]
)

In [ ]:
pd.crosstab(
    df_unstrat["group"],
    df_unstrat["os"],
    normalize="index"
)

## Анализ обычной рандомизации

In [ ]:
unstrat, delta_unstrat, se_unstrat, t_stat_unstrat, ci_unstrat = analyze_ab(df_unstrat)

print(unstrat)
print("SE", se_unstrat)
print("delta", delta_unstrat)
print("t-stat", t_stat_unstrat)
print("95% CI", ci_unstrat)

## Сравнение

In [ ]:
comparison = pd.DataFrame({
    "Stratified randomization": [delta_strat, se_strat, t_stat_strat, *ci_strat],
    "Unstratified randomization": [delta_unstrat, se_unstrat, t_stat_unstrat, *ci_unstrat]
}, index=["delta", "SE", "t-stat", "CI left", "CI right"])

comparison